# 8.1. Relative data-holder use and country-holder network

## 1. Calculate relative data-holder shares by country

This section prepares the paper-level data and calculates every data holder's relative share within each country. Holder links union the paper-level standardized holder field with holders explicitly inferred from the reviewed resource–holder dictionary, then deduplicate paper–holder pairs. The upstream field currently already contains those inferred links, so the explicit union acts as a transparent consistency check. No countries or data holders are filtered out based on frequency.

In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

# Import pandas for working with the data and Path for the file location.
import pandas as pd
from pathlib import Path

In [ ]:
# Store the location of the Excel workbook.
project_folder = REPO_ROOT

data_file = (
    DATA_ROOT
    / "interdependence_paper_supplementary_data_new.xlsx"
)

In [ ]:
# Open the paper sheet, reviewed resource-to-holder dictionary, and display names.
df = pd.read_excel(
    data_file,
    sheet_name="paper_source_data",
)
source_metadata = pd.read_excel(
    data_file,
    sheet_name="standardized_sources",
    usecols=[
        "standardized_data_resource_name",
        "standardized_data_holder_name",
        "standardized_data_holder_type",
        "standardized_data_holder_country_ISO2",
    ],
)
visualization_metadata = pd.read_excel(
    data_file,
    sheet_name="visualization_names",
    usecols=[
        "standardized_data_holder_name",
        "data_holder_for_visualization",
        "standardized_data_holder_type",
        "standardized_data_holder_country_ISO2",
    ],
)


In [ ]:
# Display the first five rows so we can inspect the original DataFrame.
df.head()

In [ ]:
# Keep only papers that are marked for use in the analysis.
df = df.loc[df["used_in_analysis"].eq(True)].copy()

In [ ]:
# Create one identifier for each paper, using the DOI or the row number when the DOI is missing.
df = df.reset_index(names="row_number")

df["paper_id"] = (
    df["doi"]
    .replace("", pd.NA)
    .fillna("row_" + df["row_number"].astype(str))
)

In [ ]:
# Split affiliation countries on both separators used in the source data.
df["countries"] = (
    df["affiliations_countries"]
    .fillna("")
    .str.upper()
    .str.split(r"[;,]", regex=True)
    .apply(lambda values: [value.strip() for value in values if value.strip()])
)

In [ ]:
# Keep the first and last affiliation countries for each paper.
df["countries"] = df["countries"].apply(
    lambda countries: list(dict.fromkeys([countries[0], countries[-1]]))
    if countries
    else []
)

In [ ]:
# Build one combined paper–holder table from both valid routes.
def split_names(value):
    if pd.isna(value):
        return []
    return list(dict.fromkeys(
        item.strip() for item in str(value).split(";") if item.strip()
    ))

# Build a one-to-one lookup from canonical holder names to reviewed figure labels.
holder_display_lookup = visualization_metadata.copy()
for column in [
    "standardized_data_holder_name",
    "data_holder_for_visualization",
]:
    holder_display_lookup[column] = (
        holder_display_lookup[column].fillna("").astype(str).str.strip()
    )

holder_display_lookup = holder_display_lookup.loc[
    holder_display_lookup["standardized_data_holder_name"].ne("")
].copy()
holder_display_lookup["data_holder_for_visualization"] = (
    holder_display_lookup["data_holder_for_visualization"]
    .where(
        holder_display_lookup["data_holder_for_visualization"].ne(""),
        holder_display_lookup["standardized_data_holder_name"],
    )
)

holder_display_conflicts = (
    holder_display_lookup
    .groupby("standardized_data_holder_name")["data_holder_for_visualization"]
    .nunique()
)
assert not holder_display_conflicts.gt(1).any()

holder_display_lookup = holder_display_lookup.drop_duplicates(
    "standardized_data_holder_name"
)
holder_display_names = holder_display_lookup.set_index(
    "standardized_data_holder_name"
)["data_holder_for_visualization"].to_dict()


# Resolve one reviewed type and host-country value for every canonical holder.
def clean_text(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).replace("\xa0", " ").split())


def first_text(values):
    for value in values:
        value = clean_text(value)
        if value:
            return value
    return ""


holder_metadata_parts = []
for metadata_priority, metadata in enumerate(
    [visualization_metadata, source_metadata]
):
    holder_part = metadata[[
        "standardized_data_holder_name",
        "standardized_data_holder_type",
        "standardized_data_holder_country_ISO2",
    ]].copy()
    holder_part["data_holder"] = holder_part[
        "standardized_data_holder_name"
    ].map(split_names)
    holder_part["metadata_specificity"] = holder_part["data_holder"].map(len)
    holder_part = holder_part.explode("data_holder").dropna(subset=["data_holder"])
    holder_part["metadata_priority"] = metadata_priority
    holder_metadata_parts.append(holder_part)

holder_metadata_lookup = (
    pd.concat(holder_metadata_parts, ignore_index=True)
    .sort_values(
        ["data_holder", "metadata_priority", "metadata_specificity"],
        kind="stable",
    )
    .groupby("data_holder", as_index=False, sort=False)
    .agg(
        holder_type=("standardized_data_holder_type", first_text),
        holder_country=("standardized_data_holder_country_ISO2", first_text),
    )
)

# Preserve the reviewed classification used in the downstream analysis.
missouri_mask = holder_metadata_lookup["data_holder"].eq(
    "Missouri Botanical Garden"
)
holder_metadata_lookup.loc[
    missouri_mask, ["holder_type", "holder_country"]
] = ["hybrid", "transnational/ambiguous"]


resource_holder_map = source_metadata[[
    "standardized_data_resource_name",
    "standardized_data_holder_name",
]].copy()
resource_holder_map["standardized_data_holder_name"] = resource_holder_map[
    "standardized_data_holder_name"
].map(split_names)
resource_holder_map = (
    resource_holder_map.explode("standardized_data_holder_name")
    .dropna(subset=["standardized_data_resource_name", "standardized_data_holder_name"])
    .drop_duplicates()
)

# Read the holders already stored in the paper-level standardized holder field.
paper_field_holder_pairs = df[["paper_id", "standardized_data_holder_names"]].copy()
paper_field_holder_pairs["data_holder"] = paper_field_holder_pairs[
    "standardized_data_holder_names"
].map(split_names)
paper_field_holder_pairs = (
    paper_field_holder_pairs.explode("data_holder")
    .dropna(subset=["data_holder"])
    [["paper_id", "data_holder"]]
)

resource_pairs = df[["paper_id", "standardized_data_resource_names"]].copy()
resource_pairs["standardized_data_resource_name"] = resource_pairs[
    "standardized_data_resource_names"
].map(split_names)
resource_pairs = resource_pairs.explode("standardized_data_resource_name")
resource_inferred_holder_pairs = (
    resource_pairs.merge(
        resource_holder_map, on="standardized_data_resource_name", how="inner"
    )
    [["paper_id", "standardized_data_holder_name"]]
    .rename(columns={"standardized_data_holder_name": "data_holder"})
)

paper_holder = pd.concat(
    [paper_field_holder_pairs, resource_inferred_holder_pairs], ignore_index=True
).drop_duplicates(["paper_id", "data_holder"])

print(f"Paper-level holder-field pairs: {len(paper_field_holder_pairs.drop_duplicates()):,}")
print(f"Resource-inferred paper–holder pairs: {len(resource_inferred_holder_pairs.drop_duplicates()):,}")
print(f"Combined unique paper–holder pairs: {len(paper_holder):,}")

In [ ]:
# Create one row for every paper-country-data-holder combination from the union.
paper_country_holder = (
    df[["paper_id", "countries"]]
    .merge(paper_holder, on="paper_id", how="inner")
    .explode("countries")
    .rename(
        columns={
            "countries": "country",
        }
    )
    .dropna(subset=["country", "data_holder"])
)

In [ ]:
# Count each paper only once for the same country and data holder.
paper_country_holder = paper_country_holder.drop_duplicates(
    subset=["paper_id", "country", "data_holder"]
)

In [ ]:
# Count the number of unique papers for every country and data holder.
relative_holder_shares = (
    paper_country_holder
    .groupby(["country", "data_holder"], as_index=False)
    .agg(holder_paper_count=("paper_id", "nunique"))
)

In [ ]:
# Calculate the total number of holder uses counted within each country.
relative_holder_shares["country_holder_use_total"] = (
    relative_holder_shares
    .groupby("country")["holder_paper_count"]
    .transform("sum")
)

In [ ]:
# Divide each holder count by its country's total to calculate the relative share.
relative_holder_shares["relative_holder_share"] = (
    relative_holder_shares["holder_paper_count"]
    / relative_holder_shares["country_holder_use_total"]
)

In [ ]:
# Sort the result and display the first 30 country-holder rows.
relative_holder_shares = relative_holder_shares.sort_values(
    by=["country", "relative_holder_share"],
    ascending=[True, False],
).reset_index(drop=True)

relative_holder_shares.head(30)

In [ ]:
# Check that the relative holder shares add up to 1 within every country.
share_check = (
    relative_holder_shares
    .groupby("country")["relative_holder_share"]
    .sum()
)

share_check.round(10).eq(1).all()

## 2. Create the country-holder bipartite network

The entire network is created in the single cell below. Economies in transition and territories not separately classified in the WESP 2026 country tables are excluded before the top-N countries are selected for Figure 2. Change the settings at the top, rerun that cell, and then rerun the visualization cells in Section 3. Providers connected to fewer than `minimum_countries_per_holder` selected countries are removed after the top-N selection.

In [ ]:
# Import NetworkX for creating the graph and NumPy for layout adjustment.
import networkx as nx
import numpy as np


# Set the number of countries and top data holders shown in the network.
number_of_countries = 50
top_holders_per_country = 10
minimum_countries_per_holder = 2

# Exclude UN WESP 2026 economies in transition from Figure 2.
excluded_transition_country_codes = {
    "AL", "AM", "AZ", "BA", "BY", "GE", "KG", "KZ", "MD", "ME",
    "MK", "RS", "RU", "TJ", "TM", "UA", "UZ",
}

# These ISO territories occur in the affiliation data but are not listed as
# separate economies in the WESP 2026 developed/developing/transition tables.
unclassified_wesp_country_codes = {
    "BM", "CW", "FO", "GL", "GP", "GU", "JE", "NC", "PF",
    "PR", "RE", "SJ", "VI", "XK",
}


# Set the reproducible spring-layout parameters.
layout_seed = 42
layout_k = 0.55
layout_iterations = 800


# Count unique papers and select the highest-paper classified non-transition economies.
country_paper_counts = (
    df[["paper_id", "countries"]]
    .explode("countries")
    .rename(columns={"countries": "country"})
    .dropna(subset=["country"])
    .groupby("country", as_index=False)
    .agg(country_paper_count=("paper_id", "nunique"))
    .sort_values("country_paper_count", ascending=False)
)

selected_countries = (
    country_paper_counts.loc[
        ~country_paper_counts["country"].isin(
            excluded_transition_country_codes | unclassified_wesp_country_codes
        )
    ]
    .head(number_of_countries)
    .copy()
)


# Create the single complete country-holder table used by Figure 2 and all
# downstream country-level analyses. Explicit zeros retain non-use.
all_countries = sorted(relative_holder_shares["country"].unique())
all_observed_holders = sorted(paper_holder["data_holder"].dropna().unique())
all_country_holder_pairs = pd.MultiIndex.from_product(
    [all_countries, all_observed_holders],
    names=["country", "data_holder"],
).to_frame(index=False)

country_holder_totals = relative_holder_shares[[
    "country", "country_holder_use_total"
]].drop_duplicates()

all_holder_country_use = (
    all_country_holder_pairs
    .merge(
        relative_holder_shares[[
            "country",
            "data_holder",
            "holder_paper_count",
            "relative_holder_share",
        ]],
        on=["country", "data_holder"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        country_holder_totals,
        on="country",
        how="left",
        validate="many_to_one",
    )
    .merge(
        country_paper_counts[["country", "country_paper_count"]],
        on="country",
        how="left",
        validate="many_to_one",
    )
    .merge(
        holder_metadata_lookup,
        on="data_holder",
        how="left",
        validate="many_to_one",
    )
)
all_holder_country_use["holder_paper_count"] = (
    all_holder_country_use["holder_paper_count"].fillna(0).astype(int)
)
all_holder_country_use["relative_holder_share"] = (
    all_holder_country_use["relative_holder_share"].fillna(0.0)
)
all_holder_country_use["data_holder_for_visualization"] = (
    all_holder_country_use["data_holder"]
    .map(holder_display_names)
    .fillna(all_holder_country_use["data_holder"])
)
scoreable_holders = set(relative_holder_shares["data_holder"])
all_holder_country_use["has_country_observation"] = (
    all_holder_country_use["data_holder"].isin(scoreable_holders)
)

assert not all_holder_country_use.duplicated(["country", "data_holder"]).any()
assert len(all_holder_country_use) == len(all_countries) * len(all_observed_holders)
assert all_holder_country_use.groupby("data_holder").size().eq(len(all_countries)).all()
assert all_holder_country_use["holder_type"].notna().all()
assert all_holder_country_use["holder_country"].notna().all()


# Keep the largest relative holder shares for each selected country.
network_edges = (
    all_holder_country_use.loc[
        all_holder_country_use["country"].isin(selected_countries["country"])
        & all_holder_country_use["holder_paper_count"].gt(0)
    ]
    .sort_values(
        ["country", "relative_holder_share", "data_holder"],
        ascending=[True, False, True],
    )
    .groupby("country", group_keys=False)
    .head(top_holders_per_country)
    .copy()
)


# Count how many selected countries are connected to each displayed data holder.
holder_country_counts = (
    network_edges
    .groupby("data_holder")["country"]
    .nunique()
)


# Remove providers connected to too few countries in this selected network.
holders_to_keep = holder_country_counts.loc[
    holder_country_counts >= minimum_countries_per_holder
].index

network_edges = network_edges.loc[
    network_edges["data_holder"].isin(holders_to_keep)
].copy()


# Record the Figure 2 holder filter on the complete table so later notebooks
# can reproduce the displayed subset without using a separate source table.
all_holder_country_use["included_in_figure_2"] = (
    all_holder_country_use["data_holder"].isin(holders_to_keep)
)


# Add reviewed display names while retaining canonical names as analytical keys.
network_edges["data_holder_for_visualization"] = (
    network_edges["data_holder"]
    .map(holder_display_names)
    .fillna(network_edges["data_holder"])
)


# Create separate internal node names for countries and data holders.
network_edges["country_node"] = "country: " + network_edges["country"]
network_edges["holder_node"] = "holder: " + network_edges["data_holder"]


# Summarize each data holder within the selected visualization network.
holder_summary = (
    network_edges
    .groupby(
        ["holder_node", "data_holder", "data_holder_for_visualization"],
        as_index=False,
    )
    .agg(
        country_count=("country", "nunique"),
        max_relative_share=("relative_holder_share", "max"),
    )
)


# Create an empty undirected NetworkX graph.
G = nx.Graph()


# Add the selected country nodes to the graph.
for row in selected_countries.itertuples(index=False):
    G.add_node(
        "country: " + row.country,
        label=row.country,
        node_type="country",
        paper_count=row.country_paper_count,
    )


# Add the retained data-holder nodes to the graph.
for row in holder_summary.itertuples(index=False):
    G.add_node(
        row.holder_node,
        label=row.data_holder_for_visualization,
        standardized_data_holder_name=row.data_holder,
        node_type="data_holder",
        country_count=row.country_count,
        max_relative_share=row.max_relative_share,
    )


# Add country-holder edges and store the original relative share on each edge.
for row in network_edges.itertuples(index=False):
    G.add_edge(
        row.country_node,
        row.holder_node,
        relative_share=row.relative_holder_share,
        holder_paper_count=row.holder_paper_count,
    )


# Store the country nodes and data-holder nodes in separate lists for drawing.
country_nodes = [
    node for node, details in G.nodes(data=True)
    if details["node_type"] == "country"
]

holder_nodes = [
    node for node, details in G.nodes(data=True)
    if details["node_type"] == "data_holder"
]


# Calculate node positions using the relative shares as layout weights.
positions = nx.spring_layout(
    G,
    weight="relative_share",
    seed=layout_seed,
    k=layout_k,
    iterations=layout_iterations,
)


# Pull only far-out data holders toward the country-network centre.
# Country nodes and centrally placed holder nodes retain their coordinates.
outer_holder_radius_threshold = 0.75
outer_holder_excess_scale = 0.35

country_layout_center = np.median(
    np.array([positions[node] for node in country_nodes]),
    axis=0,
)

adjusted_outer_holder_nodes = []
for node in holder_nodes:
    offset_from_center = positions[node] - country_layout_center
    holder_radius = np.linalg.norm(offset_from_center)

    if holder_radius > outer_holder_radius_threshold:
        adjusted_radius = (
            outer_holder_radius_threshold
            + outer_holder_excess_scale
            * (holder_radius - outer_holder_radius_threshold)
        )
        positions[node] = (
            country_layout_center
            + offset_from_center * adjusted_radius / holder_radius
        )
        adjusted_outer_holder_nodes.append(node)


# Display a short summary of the completed graph.
print(f"Countries: {len(country_nodes)}")
print(f"Data holders: {len(holder_nodes)}")
print(f"Edges: {G.number_of_edges()}")
print(f"Minimum countries per holder: {minimum_countries_per_holder}")
print(
    "Outer data holders moved inward: "
    f"{len(adjusted_outer_holder_nodes)}"
)


## 3. Visualize the country-holder network

The graph and its positions already exist before this section. You can therefore adjust the settings below and rerun only the two visualization cells.

In [ ]:
# Import the plotting tools and set the visual parameters that can be adjusted later.
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D

figure_size = (22, 18.5)

country_color = "#4E79A7"
holder_color = "#E15759"
edge_color = "#8E8E8E"

country_node_size = 900
holder_size_base = 140
holder_size_scale = 220

edge_width_minimum = 0.15
edge_width_scale = 7.0
edge_alpha_minimum = 0.06
edge_alpha_scale = 1.25

country_label_size = 16
holder_label_size = 16
provider_label_threshold = 0.02

In [ ]:
# Draw and save the bipartite network using the settings from the previous cell.
fig, ax = plt.subplots(figsize=figure_size, dpi=160)
ax.axis("off")


# Make thicker and more opaque edges represent larger relative shares.
edge_shares = [
    G.edges[edge]["relative_share"]
    for edge in G.edges
]

edge_widths = [
    edge_width_minimum + edge_width_scale * share ** 0.5
    for share in edge_shares
]

edge_colors = [
    to_rgba(
        edge_color,
        min(0.80, edge_alpha_minimum + edge_alpha_scale * share ** 0.5),
    )
    for share in edge_shares
]


# Use square-root scaling so widely used providers do not become excessively large.
holder_node_sizes = [
    holder_size_base
    + holder_size_scale * G.nodes[node]["country_count"] ** 0.5
    for node in holder_nodes
]


# Draw all edges before drawing the nodes and labels.
nx.draw_networkx_edges(
    G,
    positions,
    ax=ax,
    edge_color=edge_colors,
    width=edge_widths,
)


# Draw countries as blue circles.
nx.draw_networkx_nodes(
    G,
    positions,
    nodelist=country_nodes,
    node_color=country_color,
    node_shape="o",
    node_size=country_node_size,
    edgecolors="#4A4A4A",
    linewidths=0.8,
    ax=ax,
)


# Draw data holders as red squares.
nx.draw_networkx_nodes(
    G,
    positions,
    nodelist=holder_nodes,
    node_color=holder_color,
    node_shape="s",
    node_size=holder_node_sizes,
    edgecolors="#4A4A4A",
    linewidths=0.8,
    ax=ax,
)


# Label every selected country.
country_labels = {
    node: G.nodes[node]["label"]
    for node in country_nodes
}

nx.draw_networkx_labels(
    G,
    positions,
    labels=country_labels,
    font_size=country_label_size,
    font_color="#202020",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 1.0},
    clip_on=False,
    ax=ax,
)


# Create labels only for providers exceeding the chosen share threshold.
providers_to_label = [
    node for node in holder_nodes
    if G.nodes[node]["max_relative_share"] > provider_label_threshold
]

provider_labels = {
    node: G.nodes[node]["label"]
    for node in providers_to_label
}


# Place each provider label in the centre of its provider node.
nx.draw_networkx_labels(
    G,
    positions,
    labels=provider_labels,
    font_size=holder_label_size,
    font_color="#202020",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 1.0},
    clip_on=False,
    ax=ax,
)
ax.margins(x=0.015, y=0.035)


# Create the node and edge-width legend.
legend_items = [
    Line2D(
        [0], [0], marker="o", color="none", markerfacecolor=country_color,
        markeredgecolor="#4A4A4A", markersize=9, label="Country",
    ),
    Line2D(
        [0], [0], marker="s", color="none", markerfacecolor=holder_color,
        markeredgecolor="#4A4A4A", markersize=9, label="Data holder",
    ),
]

for example_share in [0.01, 0.05, 0.10, 0.20]:
    example_width = edge_width_minimum + edge_width_scale * example_share ** 0.5
    example_alpha = min(
        0.80,
        edge_alpha_minimum + edge_alpha_scale * example_share ** 0.5,
    )
    legend_items.append(
        Line2D(
            [0], [0],
            color=to_rgba(edge_color, example_alpha),
            linewidth=example_width,
            label=f"{example_share:.0%} relative share",
        )
    )

ax.legend(
    handles=legend_items,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.03),
    ncol=3,
    frameon=False,
    fontsize=8,
)


# Add a title describing the current adjustable selection.
ax.set_title(
    f"Top {top_holders_per_country} shared data holders for the "
    f"{number_of_countries} non-transition countries with the most papers",
    fontsize=14,
    pad=18,
)

fig.tight_layout()


# Save both a high-resolution PNG and an editable SVG.
output_folder = DATA_ROOT / "visualizations"
output_folder.mkdir(exist_ok=True)

png_file = output_folder / "country_holder_bipartite_top50_top10.png"
svg_file = output_folder / "country_holder_bipartite_top50_top10.svg"

fig.savefig(png_file, dpi=300, bbox_inches="tight")
fig.savefig(svg_file, bbox_inches="tight")

print(f"Saved: {png_file}")
print(f"Saved: {svg_file}")

plt.show()

In [ ]:
# Draw a second network version using the UN WESP 2026 economic classifications.

# Record the official source used for the economic groups.
un_classification_source = "https://desapublications.un.org/file/21554/download"


# List the ISO country codes in the UN category "developed economies".
un_developed_country_codes = {
    "AT", "AU", "BE", "BG", "CA", "CH", "CY", "CZ", "DE", "DK",
    "EE", "ES", "FI", "FR", "GB", "GR", "HR", "HU", "IE", "IS",
    "IT", "JP", "KR", "LT", "LU", "LV", "MT", "NL", "NO", "NZ",
    "PL", "PT", "RO", "SE", "SI", "SK", "US",
}


# Reuse the transition-country set excluded from the Figure 2 network.
un_transition_country_codes = excluded_transition_country_codes
un_unclassified_country_codes = unclassified_wesp_country_codes


# Use one color for each retained UN economic group.
un_developed_country_color = "#4E79A7"
un_developing_country_color = "#59A14F"


# Separate the selected country nodes into the two retained UN economic groups.
un_developed_country_nodes = [
    node for node in country_nodes
    if G.nodes[node]["label"] in un_developed_country_codes
]

un_transition_country_nodes = [
    node for node in country_nodes
    if G.nodes[node]["label"] in un_transition_country_codes
]

un_unclassified_country_nodes = [
    node for node in country_nodes
    if G.nodes[node]["label"] in un_unclassified_country_codes
]

un_developing_country_nodes = [
    node for node in country_nodes
    if G.nodes[node]["label"] not in un_developed_country_codes
    and G.nodes[node]["label"] not in un_transition_country_codes
    and G.nodes[node]["label"] not in un_unclassified_country_codes
]


# Create the figure and calculate the same edge and provider sizes as before.
fig, ax = plt.subplots(figsize=figure_size, dpi=160)
ax.axis("off")

un_edge_shares = [
    G.edges[edge]["relative_share"]
    for edge in G.edges
]

un_edge_widths = [
    edge_width_minimum + edge_width_scale * share ** 0.5
    for share in un_edge_shares
]

un_edge_colors = [
    to_rgba(
        edge_color,
        min(0.80, edge_alpha_minimum + edge_alpha_scale * share ** 0.5),
    )
    for share in un_edge_shares
]

un_holder_sizes = [
    holder_size_base
    + holder_size_scale * G.nodes[node]["country_count"] ** 0.5
    for node in holder_nodes
]


# Draw the weighted country-holder edges.
nx.draw_networkx_edges(
    G,
    positions,
    ax=ax,
    edge_color=un_edge_colors,
    width=un_edge_widths,
)


# Draw developed economies as blue circles.
nx.draw_networkx_nodes(
    G,
    positions,
    nodelist=un_developed_country_nodes,
    node_color=un_developed_country_color,
    node_shape="o",
    node_size=country_node_size,
    edgecolors="#4A4A4A",
    linewidths=0.8,
    ax=ax,
)


# Draw developing economies as green circles.
nx.draw_networkx_nodes(
    G,
    positions,
    nodelist=un_developing_country_nodes,
    node_color=un_developing_country_color,
    node_shape="o",
    node_size=country_node_size,
    edgecolors="#4A4A4A",
    linewidths=0.8,
    ax=ax,
)


# Draw data holders as red squares.
nx.draw_networkx_nodes(
    G,
    positions,
    nodelist=holder_nodes,
    node_color=holder_color,
    node_shape="s",
    node_size=un_holder_sizes,
    edgecolors="#4A4A4A",
    linewidths=0.8,
    ax=ax,
)


# Add every country label with a white background.
un_country_labels = {
    node: G.nodes[node]["label"]
    for node in country_nodes
}

nx.draw_networkx_labels(
    G,
    positions,
    labels=un_country_labels,
    font_size=country_label_size,
    font_color="#202020",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 1.0},
    clip_on=False,
    ax=ax,
)


# Add centered labels to providers that exceed the chosen share threshold.
un_provider_labels = {
    node: G.nodes[node]["label"]
    for node in holder_nodes
    if G.nodes[node]["max_relative_share"] > provider_label_threshold
}

nx.draw_networkx_labels(
    G,
    positions,
    labels=un_provider_labels,
    font_size=holder_label_size,
    font_color="#202020",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 1.0},
    clip_on=False,
    ax=ax,
)
ax.margins(x=0.015, y=0.035)


# Create a legend for the two retained UN groups, data holders, and shares.
un_legend_items = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=un_developed_country_color,
        markeredgecolor="#4A4A4A", markersize=12,
        label="Developed economy",
    ),
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=un_developing_country_color,
        markeredgecolor="#4A4A4A", markersize=12,
        label="Developing economy",
    ),
    Line2D(
        [0], [0], marker="s", color="none",
        markerfacecolor=holder_color,
        markeredgecolor="#4A4A4A", markersize=12,
        label="Data holder",
    ),
]

for example_share in [0.01, 0.05, 0.10, 0.20]:
    example_width = edge_width_minimum + edge_width_scale * example_share ** 0.5
    example_alpha = min(
        0.80,
        edge_alpha_minimum + edge_alpha_scale * example_share ** 0.5,
    )
    un_legend_items.append(
        Line2D(
            [0], [0],
            color=to_rgba(edge_color, example_alpha),
            linewidth=example_width,
            label=f"{example_share:.0%} relative share",
        )
    )

ax.legend(
    handles=un_legend_items,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.03),
    ncol=3,
    frameon=False,
    fontsize=17,
)


fig.tight_layout()


# Save this version separately so the original visualization is preserved.
un_output_name = (
    f"country_holder_bipartite_top{number_of_countries}_"
    f"top{top_holders_per_country}_un_wesp_2026"
)

un_png_file = output_folder / f"{un_output_name}.png"
un_svg_file = output_folder / f"{un_output_name}.svg"
un_pdf_file = output_folder / "Fig2a.pdf"

fig.savefig(un_png_file, dpi=360, bbox_inches="tight")
fig.savefig(un_svg_file, bbox_inches="tight")
fig.savefig(un_pdf_file, bbox_inches="tight")

print(f"Developed economies: {len(un_developed_country_nodes)}")
assert not un_transition_country_nodes
assert not un_unclassified_country_nodes
print(f"Developing economies: {len(un_developing_country_nodes)}")
print(f"UN classification source: {un_classification_source}")
print(f"Saved: {un_png_file}")
print(f"Saved: {un_svg_file}")
print(f"Saved: {un_pdf_file}")

plt.show()


### 3.1. Color the network countries by continent

This version keeps the same nodes, edges, positions, and visual settings while replacing the UN economic-group colors with continent colors.


In [ ]:
# Import country_converter to translate valid ISO country codes into UN regions.
import country_converter as coco


# Match each UN region to one of the six continent groups used here.
un_region_to_continent = {
    "Eastern Africa": "Africa",
    "Middle Africa": "Africa",
    "Northern Africa": "Africa",
    "Southern Africa": "Africa",
    "Western Africa": "Africa",
    "Central Asia": "Asia",
    "Eastern Asia": "Asia",
    "South-eastern Asia": "Asia",
    "Southern Asia": "Asia",
    "Western Asia": "Asia",
    "Eastern Europe": "Europe",
    "Northern Europe": "Europe",
    "Southern Europe": "Europe",
    "Western Europe": "Europe",
    "Caribbean": "North America",
    "Central America": "North America",
    "Northern America": "North America",
    "South America": "South America",
    "Australia and New Zealand": "Oceania",
    "Melanesia": "Oceania",
    "Micronesia": "Oceania",
    "Polynesia": "Oceania",
}


# Create a small function that preserves unsupported codes as Unclassified.
def country_code_to_continent(country_code):
    if not isinstance(country_code, str):
        return "Unclassified"

    if len(country_code) != 2 or not country_code.isalpha():
        return "Unclassified"

    un_region = coco.convert(
        names=country_code,
        src="ISO2",
        to="UNregion",
        not_found="Unclassified",
    )

    return un_region_to_continent.get(un_region, "Unclassified")


# Store the continent of every country node in the network.
network_country_continents = {
    node: country_code_to_continent(G.nodes[node]["label"])
    for node in country_nodes
}

pd.Series(network_country_continents).value_counts()


In [ ]:
# Set one color for each continent and a fallback color for unclassified codes.
continent_colors = {
    "Africa": "#59A14F",
    "Asia": "#EDC948",
    "Europe": "#4E79A7",
    "North America": "#F28E2B",
    "South America": "#B07AA1",
    "Oceania": "#76B7B2",
    "Unclassified": "#BAB0AC",
}


# Create the continent-colored figure using the existing layout and edge settings.
fig, ax = plt.subplots(figsize=figure_size, dpi=160)
ax.axis("off")

nx.draw_networkx_edges(
    G,
    positions,
    ax=ax,
    edge_color=edge_colors,
    width=edge_widths,
)


# Draw the country nodes one continent at a time.
continents_in_network = []

for continent, color in continent_colors.items():
    nodes_in_continent = [
        node
        for node in country_nodes
        if network_country_continents[node] == continent
    ]

    if nodes_in_continent:
        continents_in_network.append(continent)

        nx.draw_networkx_nodes(
            G,
            positions,
            nodelist=nodes_in_continent,
            node_color=color,
            node_shape="o",
            node_size=country_node_size,
            edgecolors="#4A4A4A",
            linewidths=0.8,
            ax=ax,
        )


# Draw data holders as the same red squares used in the other network versions.
nx.draw_networkx_nodes(
    G,
    positions,
    nodelist=holder_nodes,
    node_color=holder_color,
    node_shape="s",
    node_size=holder_node_sizes,
    edgecolors="#4A4A4A",
    linewidths=0.8,
    ax=ax,
)


# Add the same country and provider labels used in the existing network.
nx.draw_networkx_labels(
    G,
    positions,
    labels=country_labels,
    font_size=country_label_size,
    font_color="#202020",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 1.0},
    ax=ax,
)

nx.draw_networkx_labels(
    G,
    positions,
    labels=provider_labels,
    font_size=holder_label_size,
    font_color="#202020",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 1.0},
    ax=ax,
)


# Create a legend for continents, data holders, and relative shares.
continent_legend_items = [
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor=continent_colors[continent],
        markeredgecolor="#4A4A4A", markersize=9,
        label=continent,
    )
    for continent in continents_in_network
]

continent_legend_items.append(
    Line2D(
        [0], [0], marker="s", color="none",
        markerfacecolor=holder_color,
        markeredgecolor="#4A4A4A", markersize=9,
        label="Data holder",
    )
)

for example_share in [0.01, 0.05, 0.10, 0.20]:
    example_width = edge_width_minimum + edge_width_scale * example_share ** 0.5
    example_alpha = min(
        0.80,
        edge_alpha_minimum + edge_alpha_scale * example_share ** 0.5,
    )
    continent_legend_items.append(
        Line2D(
            [0], [0],
            color=to_rgba(edge_color, example_alpha),
            linewidth=example_width,
            label=f"{example_share:.0%} relative share",
        )
    )

ax.legend(
    handles=continent_legend_items,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.03),
    ncol=6,
    frameon=False,
    fontsize=8,
)


# Add a title identifying the continent colors.
ax.set_title(
    f"Top {top_holders_per_country} shared data holders for the "
    f"{number_of_countries} non-transition countries with the most papers, "
    "by continent",
    fontsize=14,
    pad=18,
)

fig.tight_layout()


# Save the continent version separately from the other network figures.
continent_network_output_name = (
    f"country_holder_bipartite_top{number_of_countries}_"
    f"top{top_holders_per_country}_continents"
)

continent_network_png_file = (
    output_folder / f"{continent_network_output_name}.png"
)

continent_network_svg_file = (
    output_folder / f"{continent_network_output_name}.svg"
)

fig.savefig(continent_network_png_file, dpi=300, bbox_inches="tight")
fig.savefig(continent_network_svg_file, bbox_inches="tight")

print(f"Saved: {continent_network_png_file}")
print(f"Saved: {continent_network_svg_file}")

plt.show()


## 4. Export the overall country-holder table and the Figure 2 view

The overall table contains every observed country � data-holder combination, including explicit zeros. Figure 2 and all later country-level analyses apply their filters to this shared source.


In [ ]:
# Store the canonical holders retained by the Figure 2 filters.
visualized_providers = sorted(holders_to_keep)

print(f"Visualized providers: {len(visualized_providers)}")
visualized_providers


In [ ]:
# Validate the complete grid before adding the shared classifications.
assert all_holder_country_use["country"].nunique() == len(all_countries)
assert all_holder_country_use["data_holder"].nunique() == len(all_observed_holders)
assert all_holder_country_use.groupby("data_holder").size().eq(len(all_countries)).all()

all_holder_country_use.head()


In [ ]:
# Confirm that the Figure 2 holder flag reproduces the selected network nodes.
figure2_flagged_holders = set(
    all_holder_country_use.loc[
        all_holder_country_use["included_in_figure_2"],
        "data_holder",
    ]
)
assert figure2_flagged_holders == set(visualized_providers)
assert figure2_flagged_holders == set(network_edges["data_holder"])

print(f"Overall countries: {len(all_countries)}")
print(f"Overall holders: {len(all_observed_holders)}")
print(f"Figure 2 holders: {len(figure2_flagged_holders)}")


In [ ]:
# Assign UN WESP 2026 groups to the complete table. Territories that are
# absent from the WESP country tables remain explicitly unclassified.
all_holder_country_use["economic_group"] = "Developing economies"

all_holder_country_use.loc[
    all_holder_country_use["country"].isin(un_unclassified_country_codes),
    "economic_group",
] = "Not classified"

all_holder_country_use.loc[
    all_holder_country_use["country"].isin(un_developed_country_codes),
    "economic_group",
] = "Developed economies"

all_holder_country_use.loc[
    all_holder_country_use["country"].isin(un_transition_country_codes),
    "economic_group",
] = "Economies in transition"


# Classify each country once and add its continent to every holder row.
country_continent_lookup = {
    country: country_code_to_continent(country)
    for country in all_holder_country_use["country"].unique()
}
all_holder_country_use["continent"] = (
    all_holder_country_use["country"].map(country_continent_lookup)
)

overall_column_order = [
    "country",
    "economic_group",
    "continent",
    "country_paper_count",
    "data_holder",
    "data_holder_for_visualization",
    "holder_type",
    "holder_country",
    "holder_paper_count",
    "country_holder_use_total",
    "relative_holder_share",
    "has_country_observation",
    "included_in_figure_2",
]
all_holder_country_use = all_holder_country_use[overall_column_order]

# Keep the former Figure 2-specific table as a compatibility export, but derive
# it only by filtering the overall table.
provider_relative_use = all_holder_country_use.loc[
    all_holder_country_use["included_in_figure_2"]
].copy()

all_holder_country_use.head()


In [ ]:
# Save the single overall source and a compatibility view for Figure 2.
overall_holder_use_file = (
    DATA_ROOT
    / "processed_data"
    / "country_relative_use_all_data_holders.csv"
)
provider_relative_use_file = (
    DATA_ROOT
    / "processed_data"
    / "country_relative_use_visualized_providers.csv"
)

all_holder_country_use.to_csv(overall_holder_use_file, index=False)
provider_relative_use.to_csv(provider_relative_use_file, index=False)

print(f"Countries: {all_holder_country_use['country'].nunique()}")
print(f"All holders: {all_holder_country_use['data_holder'].nunique()}")
print(f"Overall rows: {len(all_holder_country_use):,}")
print(f"Figure 2 holders: {provider_relative_use['data_holder'].nunique()}")
print(f"Saved: {overall_holder_use_file}")
print(f"Saved compatibility view: {provider_relative_use_file}")


## 5. Create the supplementary table of average dependence scores

This simple publication table is aggregated from the complete country-holder table exported in Section 4. Every observed country has one row per holder, so non-use contributes an explicit zero. Holders whose papers have no usable affiliation-country information remain listed with a blank score.


The publication table contains only the data-holder name, average dependence, unique-paper count, and the number of countries using the holder. It is sorted by average dependence, with unavailable scores placed last.


In [ ]:
# Aggregate the complete country-holder table to one row per holder.
number_of_countries_in_dependence_table = all_holder_country_use["country"].nunique()

# Count unique papers directly from the paper-holder table so papers linked
# to two affiliation countries are not counted twice.
holder_unique_paper_counts = (
    paper_holder
    .groupby("data_holder", as_index=False)
    .agg(unique_papers=("paper_id", "nunique"))
)

supplementary_holder_dependence = (
    all_holder_country_use
    .groupby("data_holder", as_index=False)
    .agg(
        average_dependence_score=("relative_holder_share", "mean"),
        countries_using_holder=("holder_paper_count", lambda values: values.gt(0).sum()),
        has_country_observation=("has_country_observation", "first"),
    )
    .merge(holder_unique_paper_counts, on="data_holder", how="left")
)
supplementary_holder_dependence.loc[
    ~supplementary_holder_dependence["has_country_observation"],
    "average_dependence_score",
] = pd.NA

supplementary_holder_dependence["average_dependence_percent"] = (
    100 * supplementary_holder_dependence["average_dependence_score"]
).round(4)
supplementary_holder_dependence["countries_using_holder"] = (
    supplementary_holder_dependence["countries_using_holder"].astype(int)
)
supplementary_holder_dependence = supplementary_holder_dependence.sort_values(
    ["average_dependence_score", "data_holder"],
    ascending=[False, True],
    na_position="last",
).reset_index(drop=True)

supplementary_average_dependence_table = (
    supplementary_holder_dependence[[
        "data_holder",
        "average_dependence_percent",
        "unique_papers",
        "countries_using_holder",
    ]]
    .rename(columns={
        "data_holder": "Data holder",
        "average_dependence_percent": "Average dependence (%)",
        "unique_papers": "Unique papers",
        "countries_using_holder": "Countries using holder",
    })
)

# Validate the comprehensive table and confirm that Figure 2 is unchanged.
scoreable_rows = supplementary_holder_dependence[
    "average_dependence_score"
].notna()
assert supplementary_holder_dependence["data_holder"].is_unique
assert len(supplementary_holder_dependence) == paper_holder["data_holder"].nunique()
assert scoreable_rows.sum() == relative_holder_shares["data_holder"].nunique()
assert abs(
    supplementary_holder_dependence.loc[
        scoreable_rows, "average_dependence_score"
    ].sum() - 1
) < 1e-12
assert set(network_edges["data_holder"]).issubset(
    set(supplementary_holder_dependence["data_holder"])
)
assert network_edges["country"].nunique() == 50
assert network_edges["data_holder"].nunique() == 22
assert len(network_edges) == 462

unscoreable_holders = supplementary_holder_dependence.loc[
    ~scoreable_rows, "data_holder"
].tolist()
print(f"Countries in denominator: {number_of_countries_in_dependence_table}")
print(f"Observed data holders: {len(supplementary_holder_dependence)}")
print(f"Data holders with scores: {scoreable_rows.sum()}")
print(f"Data holders without valid country information: {unscoreable_holders}")
print("Figure 2 check: 50 countries, 22 holders, and 462 edges.")

supplementary_average_dependence_table


In [ ]:
# Save the publication-ready supplementary table separately from the
# Figure 2 companion dataset created in Section 6.
supplementary_average_dependence_file = (
    DATA_ROOT
    / "processed_data"
    / "supplementary_data_holder_average_dependence_scores.csv"
)

supplementary_average_dependence_table.to_csv(
    supplementary_average_dependence_file,
    index=False,
    na_rep="",
)

print("All supplementary average-dependence checks passed.")
print(f"Saved: {supplementary_average_dependence_file}")


## 6. Calculate the average relevance of the visualized holders (Figure 2 companion data)

This section summarizes the same 22 data holders shown in the network, but calculates their scores across **all countries in the data**, not only the 50 countries displayed in the network. Countries in which a visualized holder was not used contribute zero, and every country receives equal weight. This remains a figure-specific companion dataset rather than the comprehensive supplementary table created in Section 5.


In [ ]:
# Confirm that the Figure 2 view derived from the overall Section 4 table
# of an observed country and a data holder shown in the network.
number_of_countries_in_average = provider_relative_use["country"].nunique()
number_of_visualized_holders = provider_relative_use["data_holder"].nunique()

rows_per_holder = provider_relative_use.groupby("data_holder").size()

assert rows_per_holder.eq(number_of_countries_in_average).all()
assert len(provider_relative_use) == (
    number_of_countries_in_average * number_of_visualized_holders
)
assert provider_relative_use["relative_holder_share"].between(0, 1).all()

# Each selected canonical holder must have exactly one reviewed display label.
selected_holder_labels = provider_relative_use[
    ["data_holder", "data_holder_for_visualization"]
].drop_duplicates()
assert selected_holder_labels["data_holder"].is_unique
assert selected_holder_labels["data_holder_for_visualization"].is_unique


# Average the country-level relative-use shares, including the explicit zeros
# for countries in which a visualized holder was not used.
average_relevance_scores = (
    provider_relative_use
    .groupby(
        ["data_holder", "data_holder_for_visualization"],
        as_index=False,
    )
    .agg(
        average_relevance_score=("relative_holder_share", "mean"),
    )
    .sort_values("average_relevance_score", ascending=False)
    .reset_index(drop=True)
)

average_relevance_scores.insert(
    0,
    "rank",
    range(1, len(average_relevance_scores) + 1),
)
average_relevance_scores["average_relevance_percent"] = (
    100 * average_relevance_scores["average_relevance_score"]
).round(2)
average_relevance_scores["number_of_countries"] = number_of_countries_in_average

average_relevance_scores = average_relevance_scores[
    [
        "rank",
        "data_holder",
        "data_holder_for_visualization",
        "average_relevance_score",
        "average_relevance_percent",
        "number_of_countries",
    ]
]


# Create a compact, percentage-based table for direct use in the paper.
paper_average_relevance_table = average_relevance_scores[
    ["rank", "data_holder_for_visualization", "average_relevance_percent"]
].rename(
    columns={
        "rank": "Rank",
        "data_holder_for_visualization": "Data holder",
        "average_relevance_percent": "Average relevance (%)",
    }
)

print(f"Countries included: {number_of_countries_in_average}")
print(f"Visualized data holders: {number_of_visualized_holders}")
paper_average_relevance_table


In [ ]:
# Save the complete score table so the reported values remain reproducible.
average_relevance_scores_file = (
    DATA_ROOT
    / "processed_data"
    / "visualized_data_holder_average_relevance_scores.csv"
)

average_relevance_scores.to_csv(average_relevance_scores_file, index=False)

print("All average-relevance checks passed.")
print(f"Saved: {average_relevance_scores_file}")


## 7. Add the dependency-share table to the supplementary workbook

This optional cell copies the recovered supplementary workbook to a new file, adds the CSV from Section 5 as a worksheet, and saves the result. It never overwrites the input workbook.


In [ ]:
# Add the dependency-share CSV to a copy of the supplementary workbook.
# The input workbook is never overwritten.
from pathlib import Path
from shutil import copy2

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font
from openpyxl.utils.dataframe import dataframe_to_rows


supplementary_workbook_file = (
    DATA_ROOT
    / "interdependence_paper_supplementary_data_new_recovered.xlsx"
)
dependency_share_csv_file = (
    DATA_ROOT
    / "processed_data"
    / "supplementary_data_holder_average_dependence_scores.csv"
)
updated_supplementary_workbook_file = (
    DATA_ROOT
    / "interdependence_paper_supplementary_data_new_with_dependence_scores.xlsx"
)
dependency_share_sheet_name = "average_dependence_scores"

assert supplementary_workbook_file.exists(), supplementary_workbook_file
assert dependency_share_csv_file.exists(), dependency_share_csv_file
assert updated_supplementary_workbook_file != supplementary_workbook_file

dependency_share_table = pd.read_csv(
    dependency_share_csv_file,
    keep_default_na=False,
)
expected_columns = [
    "Data holder",
    "Average dependence (%)",
    "Unique papers",
    "Countries using holder",
]
assert dependency_share_table.columns.tolist() == expected_columns

# Start from a fresh copy every time this cell is run.
copy2(supplementary_workbook_file, updated_supplementary_workbook_file)
workbook = load_workbook(updated_supplementary_workbook_file)

if dependency_share_sheet_name in workbook.sheetnames:
    del workbook[dependency_share_sheet_name]
worksheet = workbook.create_sheet(dependency_share_sheet_name)

for row in dataframe_to_rows(dependency_share_table, index=False, header=True):
    worksheet.append(row)

for cell in worksheet[1]:
    cell.font = Font(bold=True)
worksheet.freeze_panes = "A2"
worksheet.auto_filter.ref = worksheet.dimensions
worksheet.column_dimensions["A"].width = 42
worksheet.column_dimensions["B"].width = 24
worksheet.column_dimensions["C"].width = 18
worksheet.column_dimensions["D"].width = 24
for cell in worksheet["B"][1:]:
    cell.number_format = "0.0000"
for column in ("C", "D"):
    for cell in worksheet[column][1:]:
        cell.number_format = "0"

workbook.save(updated_supplementary_workbook_file)
workbook.close()

# Reopen read-only and verify the saved worksheet before reporting success.
check_workbook = load_workbook(
    updated_supplementary_workbook_file,
    read_only=True,
)
assert dependency_share_sheet_name in check_workbook.sheetnames
check_worksheet = check_workbook[dependency_share_sheet_name]
assert check_worksheet.max_row == len(dependency_share_table) + 1
assert check_worksheet.max_column == len(expected_columns)
check_workbook.close()

print(f"Saved: {updated_supplementary_workbook_file}")


## 8. Plot the data holders with the highest average dependency shares

This supplementary figure displays the 30 highest average dependence scores calculated above, using the same layout settings as the primary-research-field supplementary figure in notebook 7.

In [ ]:
from matplotlib.ticker import PercentFormatter
from textwrap import wrap


number_of_holders_to_plot = 30
dependency_share_csv_file = (
    DATA_ROOT
    / "processed_data"
    / "supplementary_data_holder_average_dependence_scores.csv"
)
supplementary_plot_folder = DATA_ROOT / "visualizations"
supplementary_plot_folder.mkdir(parents=True, exist_ok=True)

dependency_share_table = pd.read_csv(dependency_share_csv_file)
required_columns = {"Data holder", "Average dependence (%)"}
assert required_columns.issubset(dependency_share_table.columns)

highest_dependency_shares = (
    dependency_share_table
    .dropna(subset=["Average dependence (%)"])
    .nlargest(number_of_holders_to_plot, "Average dependence (%)")
    .sort_values("Average dependence (%)", ascending=False)
    .copy()
)
highest_dependency_shares["label"] = highest_dependency_shares["Data holder"].map(
    lambda value: "\n".join(wrap(value, width=18))
)

figure, axis = plt.subplots(figsize=(20, 8.7))
axis.bar(
    highest_dependency_shares["label"],
    highest_dependency_shares["Average dependence (%)"],
    color="#2f6f9f",
)
axis.set(
    xlabel="",
    ylabel="Average dependency share across countries (%)",
)
axis.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
axis.grid(axis="y", alpha=0.25)
axis.tick_params(axis="x", labelrotation=55, labelsize=9)
for tick_label in axis.get_xticklabels():
    tick_label.set_ha("right")
    tick_label.set_rotation_mode("anchor")
axis.spines[["top", "right"]].set_visible(False)

figure.tight_layout(pad=2.0)

supplementary_bar_plot_png = (
    supplementary_plot_folder
    / "SFig2_supplementary_highest_average_dependence_scores.png"
)
supplementary_bar_plot_svg = (
    supplementary_plot_folder
    / "SFig2_supplementary_highest_average_dependence_scores.svg"
)
supplementary_bar_plot_pdf = (
    supplementary_plot_folder
    / "SFig2_supplementary_highest_average_dependence_scores.pdf"
)
figure.savefig(supplementary_bar_plot_png, dpi=300, bbox_inches="tight")
figure.savefig(supplementary_bar_plot_svg, bbox_inches="tight")
figure.savefig(supplementary_bar_plot_pdf, bbox_inches="tight")
print(f"Saved: {supplementary_bar_plot_png}")
print(f"Saved: {supplementary_bar_plot_svg}")
print(f"Saved: {supplementary_bar_plot_pdf}")
plt.show()
